# rl17_update

> Phase 17, the pieces of `trainRL17`'s step: the step's board groups (`boardGroups`: team sizes from `mixN`, scripted liars on a share of the boards), the reply-level clipped policy-gradient term (`clippedTerm`) and pilot v3's fix (`valueMask`: loss and KL on value tokens only; `tokenTerm`: a per-token clipped ratio; `cutCount`: prompts the training window cuts), the update passes with the reference and old log-probabilities computed once per step (`ppoUpdate`), and the turn logs of every board group (`saveTurns`). The offline tests need no GPU.

In [ ]:
#| default_exp rl17_update

In [ ]:
#| hide
from nbdev.showdoc import *

## Board groups, the loss, the passes

In [ ]:
#| export
import json
from pathlib import Path

import numpy as np

from explRL.local_policy import promptText
from explRL.rl_torch import replyLogps


def boardGroups(cfg, s):
    """The step's board groups: [(N, nLiar, seeds)] with each board seed repeated `group` times. The step's distinct seeds
    (as stepSeeds) are split in order by mixN; within a team size the first round(liarShare x boards) carry N // 4 liars."""
    distinct = [cfg.boardSeed0 + s * cfg.seedsPerStep + j for j in range(cfg.seedsPerStep)]
    out, at = [], 0
    for k, (N, share) in enumerate(cfg.mixN):
        n = len(distinct) - at if k == len(cfg.mixN) - 1 else int(round(cfg.seedsPerStep * share))
        mine, at = distinct[at:at + n], at + n
        nl = int(round(cfg.liarShare * len(mine)))
        for liars, seeds in ((N // 4, mine[:nl]), (0, mine[nl:])):
            if seeds:
                out.append((N, liars, [x for x in seeds for _ in range(cfg.group)]))
    return out


def _valueChars(s):
    "Per character of a reply, True inside a value of its top-level JSON object (a string's content and closing quote, a number or literal); None if the reply is not an object. Tolerates a cut reply."
    i = len(s) - len(s.lstrip())
    if s[i:i + 1] != '{':
        return None
    v, i, key = [False] * len(s), i + 1, True
    while i < len(s):
        ch = s[i]
        if ch == '"':
            j = i + 1
            while j < len(s) and s[j] != '"':
                j += 2 if s[j] == '\\' else 1
            if not key:
                v[i + 1:min(j + 1, len(s))] = [True] * (min(j + 1, len(s)) - i - 1)
            i = j + 1
            continue
        if ch in ':,':
            key = ch == ','
        elif not key and ch not in ' \t\n\r}':
            v[i] = True
        i += 1
    return v


def valueMask(tok, reply, eos=True):
    """Per reply token as `encodePair` builds them (the reply's tokens, then eos unless cut): 1 if the token overlaps a value
    of the reply's JSON object, else 0: keys, punctuation, opening quotes and eos are fixed by the reply schema under
    constrained decoding, so they carry no decision (pilot v2: a free route to lower a reply's probability). A reply that
    is not a JSON object: every token 1."""
    v, enc = _valueChars(reply), tok(reply, add_special_tokens=False, return_offsets_mapping=True)
    if v is None:
        return [1] * len(enc['input_ids']) + ([1] if eos else [])
    return [int(any(v[a:b])) for a, b in enc['offset_mapping']] + ([0] if eos else [])


def pairLengths(tok, samples):
    "Tokens of each sample's prompt plus reply (and eos), as encodePair counts them before any cut."
    return [len(tok(promptText(tok, s['system'], s['blocks']), add_special_tokens=False)['input_ids'])
            + len(tok(s['reply'], add_special_tokens=False)['input_ids']) + int(s['eos']) for s in samples]


def cutCount(tok, samples, maxLen, lengths=None):
    "How many samples' prompt and reply exceed the training window (encodePair then cuts the prompt from the left)."
    return sum(n > maxLen for n in (pairLengths(tok, samples) if lengths is None else lengths))


def tokenTerm(lp, old, adv, clip, w):
    "One reply's term over its weighted tokens: -A x mean log pi without a clip, else PPO's per-token clipped ratio."
    import torch
    n = w.sum().clamp(min=1.0)
    if clip is None:
        return -adv * (lp * w).sum() / n
    r = torch.exp(lp - old)
    return -(torch.min(r * adv, torch.clamp(r, 1 - clip, 1 + clip) * adv) * w).sum() / n


def clippedTerm(lp, old, adv, clip):
    "One reply's policy-gradient term: Phase 12's -A * mean log pi without a clip, else PPO's clipped reply-level ratio."
    import torch
    if clip is None:
        return -adv * lp.mean()
    r = torch.exp(lp.mean() - old.mean())
    return -torch.min(r * adv, torch.clamp(r, 1 - clip, 1 + clip) * adv)


def ppoUpdate(model, tok, opt, params, samples, chunks, kl, maxLen, device, epochs=1, clip=None, seed=0, valueOnly=False,
              tokenClip=False):
    """`epochs` passes over `samples` (micro-batches `chunks`; passes after the first in a seeded shuffled order); the
    reference (adapter off) and old log-probabilities once, before the first pass; per pass: the clipped term plus
    kl x k3 to the reference, gradient clipping 1.0, one optimizer step. -> loss, KL, tokens (the last pass's).
    Pilot v3's fix: `valueOnly` weights the loss and KL by `valueMask` (and adds klAll, the KL over every token);
    `tokenClip` clips each token's ratio (`tokenTerm`) instead of the reply-level mean."""
    import torch
    W = [[torch.tensor(valueMask(tok, x['reply'], x['eos']), dtype=torch.float32, device=device) for x in ch]
         for ch in chunks] if valueOnly else None
    with torch.no_grad():
        ref = [replyLogps(model, tok, ch, maxLen, device) for ch in _off(model, chunks)]
        old = [[x.detach() for x in replyLogps(model, tok, ch, maxLen, device)] for ch in chunks] if clip is not None else None
    order = list(range(len(chunks)))
    for e in range(epochs):
        if e:
            np.random.default_rng([seed, e]).shuffle(order)
        stats = dict(loss=0.0, kl=0.0, tokens=0, **(dict(klAll=0.0) if valueOnly else {}))
        for c in order:
            lps, loss = replyLogps(model, tok, chunks[c], maxLen, device), 0.0
            for n, (x, lp, rf) in enumerate(zip(chunks[c], lps, ref[c])):
                d = rf - lp
                kv = torch.exp(d) - d - 1
                if valueOnly or tokenClip:
                    w = W[c][n] if valueOnly else torch.ones_like(lp)
                    assert len(w) == len(lp), 'the value mask and the reply tokens disagree'
                    k3 = (kv * w).sum() / w.sum().clamp(min=1.0)
                    term = tokenTerm(lp, old[c][n] if old else None, x['adv'], clip, w)
                else:
                    k3, term = kv.mean(), clippedTerm(lp, old[c][n] if old else None, x['adv'], clip)
                loss = loss + term + kl * k3
                stats['kl'] += k3.item() / len(samples)
                if valueOnly:
                    stats['klAll'] += kv.mean().item() / len(samples)
                stats['tokens'] += len(lp)
            (loss / len(samples)).backward()
            stats['loss'] += loss.item() / len(samples)
        if samples:
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            opt.step()
        opt.zero_grad()
    return stats


def saveTurns(out, step, gs):
    "out/turns/step-<step>.jsonl: every board group's turn logs (team size and liars on each line)."
    (Path(out) / 'turns').mkdir(exist_ok=True)
    (Path(out) / 'turns' / f'step-{step:04d}.jsonl').write_text(''.join(
        json.dumps(dict(board=b, seed=ro.seeds[b], N=c.N, nLiar=c.nLiar, **x), sort_keys=True) + '\n'
        for c, ro in gs for b in range(len(ro.boards)) for lg in ro.logs[b] for x in lg))


def _off(model, chunks):
    "Yield the chunks with the model's adapter switched off (the warm start: the reference)."
    for ch in chunks:
        with model.disable_adapter():
            yield ch

## Tests

In [ ]:
import numpy as np
from explRL.rl17_update import *
from explRL.rl17_grid import RL17Config
from explRL.clue_rl import stepSeeds

# board groups: the defaults are stepSeeds; the mix splits seeds in order and puts liars on the first board of each size
c0 = RL17Config('m', 'o')
assert boardGroups(c0, 3) == [(4, 0, stepSeeds(c0, 3))]
cm = RL17Config('m', 'o', mixN=((4, 0.5), (8, 0.5)), liarShare=0.25)
g = boardGroups(cm, 0)
assert [(N, nl, len(sd)) for N, nl, sd in g] == [(4, 1, 4), (4, 0, 12), (8, 2, 4), (8, 0, 12)]
assert sorted(set(x for _, _, sd in g for x in sd)) == sorted(set(stepSeeds(cm, 0)))      # the same board seeds as stepSeeds
assert all(sd == [x for x in sd[::cm.group] for _ in range(cm.group)] for _, _, sd in g)              # each board played group times
# liar slots are never trained: on an 8-agent board with 2 scripted liars, every sample comes from an honest slot
from explRL.clue_agents import ClueLiar
from explRL.clue_grid import BOARD, CLUE_SETTINGS
from explRL.clue_policy import scriptedPolicy
from explRL.clue_roll import rollout
from explRL.llm_runs import LLMConfig
from explRL.rl_adv import stepSamples
lc = LLMConfig(task='clue', channel='blackboard', N=8, nLiar=2, turns=3, model='m', world={k: CLUE_SETTINGS[k] for k in BOARD})
ro = rollout(lc, [5, 5], scriptedPolicy(lambda s, b: '{"action": "wait", "target": "", "answer": 0, "clue": "", "notes": ""}'),
             {0: ClueLiar, 1: ClueLiar}, keepPrompts=True)
assert all(len(ro.logs[b][i]) == 0 for b in range(2) for i in (0, 1))
samp = stepSamples(ro, np.ones((2, 8)))
assert samp and all(x['slot'] >= 2 for x in samp)

In [ ]:
# pilot v3's value mask: values are 1; keys, punctuation, opening quotes and eos are 0 (a word-piece stand-in tokenizer)
import re
from explRL.rl17_update import _valueChars


class Pieces:
    "Tokens: runs of letters or digits, single other characters, with offsets (what a fast tokenizer returns)."
    eos_token_id = 0

    def __call__(self, s, add_special_tokens=False, return_offsets_mapping=False):
        m = [x.span() for x in re.finditer(r'[A-Za-z0-9]+| ?"",?|[^A-Za-z0-9]', s)]
        return dict(input_ids=list(range(1, len(m) + 1)), offset_mapping=m)


rep = '{"notes": "x mod 7", "action": "submit", "target": "", "answer": 18, "clue": ""}'
v = _valueChars(rep)
assert ''.join(c for c, f in zip(rep, v) if f) == 'x mod 7"submit""18"'       # contents plus closing quotes, the number
tk = Pieces()
pieces = [rep[a:b] for a, b in tk(rep)['offset_mapping']]
m = valueMask(tk, rep, eos=True)
assert len(m) == len(pieces) + 1 and m[-1] == 0                                 # eos: fixed by the schema
on = {p for p, f in zip(pieces, m) if f}
assert {'x', 'mod', '7', 'submit', '18', ' "",', ' ""'} <= on and not on & {'notes', 'action', 'target', 'answer', 'clue', '{', ':'}
assert valueMask(tk, 'not json', eos=True) == [1] * (len(tk('not json')['input_ids']) + 1)   # unconstrained text: all count
assert _valueChars('{"notes": "cut mid-va') is not None and _valueChars('{"notes": "cut mid-va')[-1]  # a cut reply
assert valueMask(tk, rep, eos=False) == m[:-1]


# the loss terms on hand-made numbers (torch only)
try:
    import torch
    lp, old = torch.tensor([-1.0, -2.0]), torch.tensor([-1.5, -2.5])          # mean ratio exp(0.5)
    assert torch.isclose(clippedTerm(lp, None, 2.0, None), torch.tensor(3.0))   # -A * mean log pi
    r = np.exp(0.5)
    assert torch.isclose(clippedTerm(lp, old, 2.0, 0.2), torch.tensor(-1.2 * 2.0))           # A > 0: clipped at 1 + eps
    assert torch.isclose(clippedTerm(lp, old, -2.0, 0.2), torch.tensor(float(r * 2.0)))      # A < 0: the unclipped ratio
    assert torch.isclose(clippedTerm(lp, lp, 2.0, 0.2), torch.tensor(-2.0))                  # ratio 1
    one = torch.ones(2)
    assert torch.isclose(tokenTerm(lp, None, 2.0, None, one), clippedTerm(lp, None, 2.0, None))   # no clip: as before
    # per token: a token that fell far is clipped on its own (the reply mean would let it fall clip x length nats)
    lo, od = torch.tensor([-10.0, -0.1]), torch.tensor([-0.1, -0.1])
    assert torch.isclose(tokenTerm(lo, od, -1.0, 0.2, one), torch.tensor((0.8 + 1.0) / 2))       # A < 0: no push below 0.8
    assert torch.isclose(tokenTerm(lo, od, -1.0, 0.2, torch.tensor([0.0, 1.0])), torch.tensor(1.0))   # masked token ignored
    g = lo.clone().requires_grad_(True)
    tokenTerm(g, od, -1.0, 0.2, one).backward()
    assert g.grad[0] == 0                                                                # the fallen token gets no gradient
except ImportError:
    pass

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()